# Big Data Analytics - Book Recommendation with Implicit Feedback

Thomas Timmermans - 2365790

Milan Muroni - 2262576

**Notatie (consistent in de hele notebook)**

| Symbool | Betekenis | Methode |
|---|---|---|
| `K` | aantal buren per boek (neighborhood size) | item–item |
| `k` | lengte van de aanbevelingslijst (top-k), standaard **10** | alle methodes |
| `beta` | shrinkage-drempel voor kleine overlap | item–item |
| `d` | aantal latente factoren | MF |
| `reg` | regularisatiesterkte λ | MF |
| `alpha` | confidence-gewicht voor geobserveerde likes | MF |
| `iterations` | aantal ALS-iteraties | MF |

In [1]:
%pip install scikit-learn implicit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import implicit

In [3]:
import json

file = "../datasets/books/Books.jsonl"
likes = []

with open(file, 'r') as fp:
    for i, line in enumerate(fp):
        data = json.loads(line.strip())
        # Alleen ratings strikt boven 4.0 tellen als 'like' (x_ui = 1)
        if data.get('rating', 0) > 4.0:
            likes.append((data['user_id'], data['parent_asin']))
        
print(f"Total likes: {len(likes)}")

df = pd.DataFrame(likes, columns=['user_id', 'book_id']).drop_duplicates()




Total likes: 20391478


## data inladen en filteren

In [4]:


print(f"Total unique likes: {len(df)}")

# 1. Filter op actieve users en boeken (bijv. minimaal 15 likes per user/boek)
# Pas '15' aan (bijv. naar 10 of 20) tot je totaal tussen de 100.000 en 300.000 likes zit
min_likes = 15
while True:
    n_before = len(df)
    df = df[df['user_id'].map(df['user_id'].value_counts()) >= min_likes]
    df = df[df['book_id'].map(df['book_id'].value_counts()) >= min_likes]
    if len(df) == n_before:
        break

print(f"Total likes after filtering: {len(df)}")

# 2. Zet tekst-ID's om naar simpele getallen (0, 1, 2, ...) voor de matrix
df['user_idx'] = df['user_id'].astype('category').cat.codes
df['book_idx'] = df['book_id'].astype('category').cat.codes

# 3. Bereken de statistieken (zoals gevraagd in de opdracht)
n_users = df['user_idx'].nunique()
n_books = df['book_idx'].nunique()
n_likes = len(df)
sparsity = 1 - (n_likes / (n_users * n_books))

print(f"Users: {n_users}")
print(f"Books: {n_books}")
print(f"Positive interactions: {n_likes}")
print(f"Sparsity: {sparsity:.4%}")

# 4. Maak de sparse matrix X (voorkomt een te grote dense matrix)
X = csr_matrix((np.ones(n_likes), (df['user_idx'], df['book_idx'])), shape=(n_users, n_books))

Total unique likes: 20155617
Total likes after filtering: 148608
Users: 5506
Books: 3638
Positive interactions: 148608
Sparsity: 99.2581%


# PART 1

## neighborhood-based

In [5]:
#1
likes_per_user = np.asarray(X.sum(axis=1)).ravel()
likes_per_book = np.asarray(X.sum(axis=0)).ravel()

print(f"Users: {n_users} | Books: {n_books} | Likes: {n_likes}")
print(f"Sparsity: {sparsity:.4%}")
print("\nLikes per user (min / mediaan / gemiddelde / max):", 
      likes_per_user.min(), np.median(likes_per_user), round(likes_per_user.mean(), 1), likes_per_user.max())
print("Likes per book (min / mediaan / gemiddelde / max):", 
      likes_per_book.min(), np.median(likes_per_book), round(likes_per_book.mean(), 1), likes_per_book.max())

Users: 5506 | Books: 3638 | Likes: 148608
Sparsity: 99.2581%

Likes per user (min / mediaan / gemiddelde / max): 15.0 21.0 27.0 239.0
Likes per book (min / mediaan / gemiddelde / max): 15.0 27.0 40.8 406.0


In [6]:
#2

# we transponeren de matrix zodat we de books kunnen vergelijken (die staan nu in de rijen, cosine_similarity werkt op rijen)
S = csr_matrix(cosine_similarity(X.T, dense_output=False))

print(f"Shape of similarity matrix: {S.shape}")
# Zet de diagonaal op 0 (een boek hoeft niet op zichzelf te lijken)
S.setdiag(0)
S.eliminate_zeros()

# Tel per boekenpaar hoe vaak ze SAMEN zijn geliket, en zet de diagonaal op 0
co_counts = (X.T @ X).tocsr()
co_counts.setdiag(0)

# Stel de drempelwaarde in en maak een kopie van de tabel voor de straffactor
beta = 10
shrinkage_weight = co_counts.copy()

# Bereken een weging tussen 0 en 1 (lage overlap = zware straf) en pas die toe op S
shrinkage_weight.data = shrinkage_weight.data / (shrinkage_weight.data + beta)
S_clean = S.multiply(shrinkage_weight)

# Print de matrices om te kijken of alles er goed uitziet (voor de eerste 10 rijen en kolommen)
print("X.T:")
print(X.T[:10, :10].toarray())

print("\nS_clean:")
print(S_clean[:10, :10].toarray().round(2))

Shape of similarity matrix: (3638, 3638)
X.T:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]

S_clean:
[[0.   0.   0.01 0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.01 0.   0.   0.   0.   0.   0.   0.  ]
 [0.01 0.01 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.32 0.37 0.35 0.29 0.29 0.22]
 [0.   0.   0.   0.32 0.   0.28 0.43 0.34 0.21 0.22]
 [0.   0.   0.   0.37 0.28 0.   0.37 0.31 0.42 0.22]
 [0.   0.   0.   0.35 0.43 0.37 0.   0.37 0.3  0.2 ]
 [0.   0.   0.   0.29 0.34 0.31 0.37 0.   0.31 0.21]
 [0.   0.   0.   0.29 0.21 0.42 0.3  0.31 0.   0.36]
 [0.   0.   0.   0.22 0.22 0.22 0.2  0.21 0.36 0.  ]]


In [7]:
#3
def recommend_books(user_idx, X, S_clean, K=20, top_n=10):

    # Maak een kopie van S_clean, zodat we de originele matrix niet overschrijven
    S_k = S_clean.tolil()

    # Loop door elk boek en bewaar alleen de K buren met de hoogste similarity-score (de rest wordt gewist)
    for i in range(S_k.shape[0]):
        if len(S_k.data[i]) > K:
            top_idx = np.sort(np.argsort(S_k.data[i])[-K:])
            S_k.data[i] = [S_k.data[i][j] for j in top_idx]
            S_k.rows[i] = [S_k.rows[i][j] for j in top_idx]

    # Pak alleen de rijen uit S_k van boeken die de user leuk vond, en tel ze per kolom op
    liked_books = X[user_idx].indices
    scores = np.asarray(S_k[liked_books].sum(axis=0)).ravel()

    # de boeken die de user al leuk vond, moeten we uitsluiten van de aanbevelingen met socore -1
    scores[X[user_idx].indices] = -1

    # sorteren van klein naar groot , omdraaien , en de eerste 5 terug geven
    top_books = np.argsort(scores)[::-1][:top_n]
    
    return top_books, scores[top_books]

# Test the influence of neighborhood size K on user 0
for K in [5, 20, 100]:
    books, scores = recommend_books(user_idx=0, X=X, S_clean=S_clean, K=K, top_n=10)
    print(f"K={K} -> Books: {books} | Scores: {np.round(scores, 2)}")

K=5 -> Books: [2450 2905 1006 2723  651 1694 1479 1235 1980 2344] | Scores: [2.35 1.8  1.75 1.43 1.38 1.34 1.32 1.27 1.25 1.23]
K=20 -> Books: [1479 1970 2207 2031 1785 2344 1006 2723 2450  651] | Scores: [6.39 5.47 5.47 5.46 5.37 4.83 3.6  3.53 3.37 3.37]
K=100 -> Books: [1479 2207 2031 1785 1970 2723 2344 1006  651 2422] | Scores: [8.13 8.09 8.02 7.82 7.73 7.64 7.47 6.53 5.95 5.94]


## model-based approach

In [8]:
import implicit

def recommend_mf_simple(user_idx, X, d=20, reg=0.1, alpha=40, iterations=10, top_n=10):
    # 1. Maak het ALS-model aan
    model = implicit.als.AlternatingLeastSquares(
        factors=d,
        regularization=reg,
        iterations=iterations,
        random_state=42
    )
    
    # 2. Train het model (alpha verwerk je door X met alpha te vermenigvuldigen)
    model.fit(X * alpha, show_progress=False)
    
    # 3. Vraag de top-N boeken op (filtert al gelezen boeken uit X automatisch weg!)
    top_books, scores = model.recommend(userid=user_idx, user_items=X[user_idx], N=top_n)
    
    return top_books, scores

# Test de invloed van d en reg op user 0 (nu in 1 seconde klaar i.p.v. vastlopen!)
for d in [10, 50]:
    for reg in [0.01, 100.0]:
        books, scores = recommend_mf_simple(user_idx=0, X=X, d=d, reg=reg, top_n=10)
        print(f"d={d}, reg={reg} -> Books: {books} | Scores: {np.round(scores, 2)}")

c:\Users\thoma\AppData\Local\Programs\Python\Python311\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


d=10, reg=0.01 -> Books: [3029 2160  545  804 1938 1857 2665 1719  729 2634] | Scores: [1.31 1.31 1.29 1.27 1.22 1.2  1.2  1.2  1.19 1.19]
d=10, reg=100.0 -> Books: [2634  545 2597 1432 1785 2360 2160 1694  804 1559] | Scores: [1.26 1.25 1.2  1.19 1.19 1.14 1.13 1.13 1.13 1.12]
d=50, reg=0.01 -> Books: [1223 1481 1349  804 1694 1928  552 1559  729  838] | Scores: [1.7  1.42 1.38 1.29 1.26 1.26 1.24 1.24 1.19 1.18]
d=50, reg=100.0 -> Books: [ 804  729 1481 2784 1694 1928 2360 1938 2450 1223] | Scores: [1.33 1.28 1.28 1.27 1.24 1.23 1.22 1.22 1.21 1.16]


## Popularity-based baseline

In [9]:
def recommend_popularity(user_idx, X_train, top_n=10):
    # 1. Tel het aantal likes per boek in de trainingdata
    book_popularity = np.asarray(X_train.sum(axis=0)).ravel().astype(float)
    
    # 2. Sluit boeken uit die de gebruiker al in de trainingdata heeft geliket
    liked_books = X_train[user_idx].indices
    book_popularity[liked_books] = -1
    
    # 3. Sorteer van hoog naar laag en pak de top_n
    top_books = np.argsort(book_popularity)[::-1][:top_n]
    
    return top_books, book_popularity[top_books]

# Test op user 0 (net als bij jullie andere modellen)
books, scores = recommend_popularity(user_idx=0, X_train=X, top_n=10)
print(f"Popularity -> Books: {books} | Likes: {scores.astype(int)}")

Popularity -> Books: [1785 1459 1565 1363 2723 2031 1734 1970 1006 2654] | Likes: [406 327 294 288 285 276 272 270 264 254]


# PART 2

##  Prepare train, validation and test interactions.

In [10]:
# Stap 1: per user 1 willekeurige like naar test, 1 naar validatie, de rest naar train
df_split = df.sample(frac=1, random_state=42)          # likes schudden (reproduceerbaar)
rank = df_split.groupby('user_idx').cumcount()          # 0, 1, 2, ... per user
df_split['split'] = np.where(rank == 0, 'test', np.where(rank == 1, 'val', 'train'))


def to_matrix(part):
    # 1. Pak alleen de rijen die horen bij de gekozen groep ('train', 'val' of 'test')
    d = df_split[df_split['split'] == part]
    
    # 2. Bouw een geheugenzuinig rooster (alle gebruikers x alle boeken) 
    #    en zet een '1' op elke plek waar een user een boek heeft geliked (de rest blijft 0)
    return csr_matrix((np.ones(len(d)), (d['user_idx'], d['book_idx'])), shape=(n_users, n_books))

X_train = to_matrix('train')
X_val = to_matrix('val')
X_test = to_matrix('test')

print(df_split['split'].value_counts())
print("Boeken zonder train-like:", (X_train.sum(axis=0) == 0).sum())

split
train    137596
test       5506
val        5506
Name: count, dtype: int64
Boeken zonder train-like: 0


## Evaluate recommendation quality

In [ ]:
# Stap 2: de drie methodes voor alle users tegelijk

# --- Popularity ---
def recommend_pop_all(X_train, X_hist, k=10):
    pop = np.asarray(X_train.sum(axis=0)).ravel()
    scores = np.tile(pop, (X_hist.shape[0], 1))
    scores[X_hist.nonzero()] = -np.inf                  # al gelikete boeken uitsluiten
    return np.argsort(-scores, axis=1)[:, :k]

# --- Item-item (zelfde berekening als cel #2 en #3, nu op X_train) ---
def similarity(X_train, beta=10):
    S = csr_matrix(cosine_similarity(X_train.T, dense_output=False))
    S.setdiag(0)
    co = (X_train.T @ X_train).tocsr()
    co.setdiag(0)
    co.data = co.data / (co.data + beta)                # shrinkage
    return csr_matrix(S.multiply(co))

def keep_top_K(S, K):
    S_k = S.tolil()
    for i in range(S_k.shape[0]):
        if len(S_k.data[i]) > K:
            top_idx = np.sort(np.argsort(S_k.data[i])[-K:])
            S_k.data[i] = [S_k.data[i][j] for j in top_idx]
            S_k.rows[i] = [S_k.rows[i][j] for j in top_idx]
    return S_k.tocsr()

def recommend_knn_all(X_hist, S_K, k=10):
    scores = (X_hist @ S_K).toarray()                   # som van similarities met gelikete boeken
    scores[X_hist.nonzero()] = -np.inf
    return np.argsort(-scores, axis=1)[:, :k]

# --- Matrix factorization (ALS) ---
def recommend_als_all(X_train, X_hist, d=50, reg=1.0, alpha=40, iterations=10, k=10):
    model = implicit.als.AlternatingLeastSquares(
        factors=d, regularization=reg, iterations=iterations, random_state=42)
    model.fit(X_train * alpha, show_progress=False)
    # recalculate_user=True: user-vector opnieuw berekenen uit X_hist (zodat val-likes meetellen in test)
    books, _ = model.recommend(np.arange(n_users), X_hist * alpha, N=k, recalculate_user=True)
    return books


# Snelle controle met history = train
S_train = similarity(X_train)
recs = {
    'Popularity': recommend_pop_all(X_train, X_train),
    'Item-item (K=20)': recommend_knn_all(X_train, keep_top_K(S_train, 20)),
    'ALS (d=50)': recommend_als_all(X_train, X_train, d=50, reg=1.0),
}
rows = np.repeat(np.arange(n_users), 10)
for name, r in recs.items():
    already_liked = X_train[rows, r.ravel()].sum()
    print(f"{name}: shape {r.shape}, al gelikete boeken in aanbevelingen: {int(already_liked)}")